# Phase 1: NLP EDA + Preprocessing

This notebook analyzes the balanced CFPB complaint sample and prepares text fields for two downstream modeling paths:

- `text_transformer`: lightly cleaned text for transformer fine-tuning.
- `text_ml_clean`: normalized text for TF-IDF and baseline classical ML models.

Important: transformer text should stay relatively raw. We preserve casing, punctuation, and wording there.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

from src.preprocessing.text_preprocessor import add_text_features

sns.set_theme(style='whitegrid')
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cfpb_sample_90k.csv'
CLEAN_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cfpb_sample_90k_clean.csv'


In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df.shape


In [ ]:
df.head(3)


## Dataset Health Checks

In [ ]:
required_cols = ['Product', 'Issue', 'Consumer complaint narrative']
df[required_cols].isna().sum().sort_values(ascending=False)


In [ ]:
df['Product'].value_counts()


In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, y='Product', order=df['Product'].value_counts().index)
plt.title('Product Distribution')
plt.xlabel('Rows')
plt.ylabel('Product')
plt.tight_layout()
plt.show()


## Text Feature Engineering

In [ ]:
df = add_text_features(df)
df[['text_raw', 'text_transformer', 'text_ml_clean', 'char_count', 'word_count', 'approx_token_count']].head(2)


In [ ]:
df[['char_count', 'word_count', 'approx_token_count']].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(2)


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['word_count'], bins=60)
plt.title('Complaint Word Count Distribution')
plt.xlabel('Word count')
plt.ylabel('Complaints')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11, 6))
sns.boxplot(data=df, x='word_count', y='Product', showfliers=False)
plt.title('Word Count by Product')
plt.xlabel('Word count')
plt.ylabel('Product')
plt.tight_layout()
plt.show()


## Issues and Duplicate Narratives

In [ ]:
df['Issue'].value_counts().head(20)


In [ ]:
duplicate_rows = int(df['is_duplicate_text'].sum())
duplicate_groups = int(df.loc[df['is_duplicate_text'], 'text_raw'].nunique())
duplicate_rows, duplicate_groups


## Top Keywords by Product

In [ ]:
def top_terms(group, top_n=15):
    vectorizer = CountVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2), min_df=5)
    matrix = vectorizer.fit_transform(group['text_ml_clean'].fillna(''))
    counts = matrix.sum(axis=0).A1
    terms = vectorizer.get_feature_names_out()
    return sorted(zip(terms, counts), key=lambda x: x[1], reverse=True)[:top_n]

keywords = {product: top_terms(group) for product, group in df.groupby('Product')}
keywords


## Save Cleaned Dataset

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
CLEAN_PATH
